# Vani-Kanoon Legal LLM Training (Small Model)

Uses **TinyLlama-1.1B** which fits in Kaggle's free GPU.

## Setup:
1. Upload `training_data.json` as dataset
2. Enable **GPU T4 x2**
3. Enable **Internet**
4. Run all cells

In [ ]:
# Install packages
!pip install -q transformers datasets accelerate peft bitsandbytes

In [ ]:
# Clear any existing GPU memory
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
import json
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

## 1. Load Data

In [ ]:
# Try different paths
paths = [
    "/kaggle/input/vani-kanoon-training/training_data.json",
    "/kaggle/input/training_data.json",
    "training_data.json",
]

raw_data = None
for path in paths:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        print(f"Loaded from: {path}")
        break
    except:
        continue

if raw_data is None:
    raise FileNotFoundError("training_data.json not found! Please upload it.")

print(f"Loaded {len(raw_data)} examples")

In [ ]:
# Format for training
def format_example(ex):
    instruction = ex.get('instruction', '')
    inp = ex.get('input', '')
    out = ex.get('output', '')
    
    if inp:
        text = f"<|user|>\n{instruction}\n\nInput: {inp}\n<|assistant|>\n{out}"
    else:
        text = f"<|user|>\n{instruction}\n<|assistant|>\n{out}"
    return {"text": text}

dataset = Dataset.from_list([format_example(ex) for ex in raw_data])
print(f"Dataset ready: {len(dataset)} examples")

## 2. Load Small Model (TinyLlama 1.1B)

In [ ]:
# TinyLlama - fits easily in 16GB GPU
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")

In [ ]:
# Load model in FP16 (no quantization needed for small model)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False

print(f"Model loaded: {model.num_parameters():,} parameters")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 3. Add LoRA

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

## 4. Tokenize

In [ ]:
def tokenize(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Tokenized: {len(tokenized)} examples")

## 5. Train

In [ ]:
args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=20,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Ready to train!")

In [ ]:
print("="*50)
print("TRAINING STARTED")
print("="*50)

trainer.train()

print("="*50)
print("TRAINING COMPLETE!")
print("="*50)

## 6. Save

In [ ]:
SAVE_PATH = "./vani-kanoon-legal"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Saved to {SAVE_PATH}")

In [ ]:
# Create downloadable zip
import shutil
import os

shutil.make_archive("vani-kanoon-legal", 'zip', SAVE_PATH)
size = os.path.getsize("vani-kanoon-legal.zip") / (1024*1024)
print(f"Created: vani-kanoon-legal.zip ({size:.1f} MB)")

## 7. Test

In [ ]:
def ask(question):
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("<|assistant|>")[-1].strip()

In [ ]:
questions = [
    "What is the punishment for murder under BNS 2023?",
    "How to file an FIR?",
    "What are grounds for divorce in India?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)[:500]}...")
    print("-"*50)

## Done!

Download `vani-kanoon-legal.zip` from the **Output** tab on the right.

### To use locally:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = PeftModel.from_pretrained(base, "./vani-kanoon-legal")
tokenizer = AutoTokenizer.from_pretrained("./vani-kanoon-legal")
```